In [6]:
"""
Lab 10 Starter Code
Model Optimization & Feature Engineering
"""
import numpy as np
import pandas as pd
import optuna
import time
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score, KFold
from sklearn.tree import DecisionTreeClassifier

# =========================
# LOAD DATA
# =========================
def load_data(path="lab10_data.csv"):
    df = pd.read_csv(path)
    X = df.drop(columns=["target"])
    y = df["target"]
    return X, y


# =========================
# PART A: GRID SEARCH
# =========================
def run_grid_search(X, y):
    """
    TODO:
    - Define param_grid
    - Use GridSearchCV
    - Return best estimator
    """
    param_grid = {
        'max_depth': [3, 5, 10, None],
        'min_samples_split': [2, 5, 10]
    }
    grid_search = GridSearchCV(DecisionTreeClassifier(), param_grid, cv=5)
    grid_search.fit(X, y)
    print(f"Grid Search Best Params: {grid_search.best_params_}")
    return grid_search.best_estimator_


# =========================
# PART A: RANDOM SEARCH
# =========================
def run_random_search(X, y):
    """
    TODO:
    - Define param_dist
    - Use RandomizedSearchCV
    - Return best estimator
    """
    param_dist = {
        'max_depth': [int(x) for x in np.linspace(3, 20, 10)],
        'min_samples_split': [2, 5, 10, 20]
    }
    start_time = time.time()
    random_search = RandomizedSearchCV(DecisionTreeClassifier(), param_dist, n_iter=10, cv=5)
    random_search.fit(X, y)
    duration = time.time() - start_time
    print(f"Random Search Best Params: {random_search.best_params_} (Runtime: {duration:.2f}s)")
    return random_search.best_estimator_


# =========================
# PART A: OPTUNA
# =========================
def run_optuna(X, y):
    """
    TODO:
    - Define objective
    - Run study.optimize
    - Return best params
    """
    def objective(trial):
        max_depth = trial.suggest_int("max_depth", 2, 32, log=True)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)

        model = DecisionTreeClassifier(max_depth=max_depth, min_samples_split=min_samples_split)
        score = cross_val_score(model, X, y, n_jobs=-1, cv=3).mean()
        return score

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=20)
    print(f"Optuna Best Params: {study.best_params}")
    return study.best_params


# =========================
# PART B: CROSS VALIDATION
# =========================
def evaluate_cv(model, X, y):
    """
    Returns:
        mean_accuracy, std_accuracy
    """
    scores = cross_val_score(model, X, y, cv=5)
    print(f"Mean Accuracy: {scores.mean():.4f}")
    print(f"Standard Deviation: {scores.std():.4f}")
    return scores.mean(), scores.std()


# =========================
# PART C: TARGET ENCODING
# =========================
def target_encode(df, column, target):
    """
    Naive target encoding (leakage-prone)
    """
    encoding = df.groupby(column)[target].mean()
    return df[column].map(encoding)


def kfold_target_encode(df, column, target, n_splits=5):
    """
    Leakage-safe encoding
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    encoded_column = pd.Series(index=df.index, dtype=float)

    for train_idx, val_idx in kf.split(df):
        train_fold_mean = df.iloc[train_idx].groupby(column)[target].mean()
        encoded_column.iloc[val_idx] = df.iloc[val_idx][column].map(train_fold_mean)

    return encoded_column.fillna(df[target].mean())

# =========================
# PART E: CYCLICAL FEATURES
# =========================
def encode_cyclical(feature, max_val):
    """
    Returns:
        sin_feature, cos_feature
    """
    sin_feat = np.sin(2 * np.pi * feature / max_val)
    cos_feat = np.cos(2 * np.pi * feature / max_val)
    return sin_feat, cos_feat




def main():
    try:
        X, y = load_data("lab10_data.csv")
        print("Data loaded successfully.\n")
    except FileNotFoundError:
        print("Error: lab10_data.csv not found.")
        return

    if 'hour' in X.columns:
        print("Engineering Cyclical Features...")
        X['hour_sin'], X['hour_cos'] = encode_cyclical(X['hour'], 24)
        X = X.drop(columns=['hour'])

    if 'city' in X.columns:
        print("Applying K-Fold Target Encoding to 'city'...")
        df_temp = X.copy()
        df_temp['target'] = y
        X['city_encoded'] = kfold_target_encode(df_temp, 'city', 'target')
        X = X.drop(columns=['city'])

    print("\n--- Running Grid Search ---")
    best_grid_model = run_grid_search(X, y)

    print("\n--- Running Random Search ---")
    best_random_model = run_random_search(X, y)

    print("\n--- Running Optuna (Bayesian Optimization) ---")
    best_optuna_params = run_optuna(X, y)

    final_model = DecisionTreeClassifier(**best_optuna_params)
    final_model.fit(X, y)

    print("\n--- Final Model Evaluation (5-Fold CV) ---")
    evaluate_cv(final_model, X, y)

if __name__ == "__main__":
    main()

Data loaded successfully.

Engineering Cyclical Features...
Applying K-Fold Target Encoding to 'city'...

--- Running Grid Search ---
Grid Search Best Params: {'max_depth': 3, 'min_samples_split': 2}

--- Running Random Search ---


[I 2026-04-23 04:59:54,913] A new study created in memory with name: no-name-0ca77e80-1d2d-44a5-a8bd-9d52bf3abdc8
[I 2026-04-23 04:59:55,017] Trial 0 finished with value: 1.0 and parameters: {'max_depth': 5, 'min_samples_split': 8}. Best is trial 0 with value: 1.0.


Random Search Best Params: {'min_samples_split': 10, 'max_depth': 14} (Runtime: 0.80s)

--- Running Optuna (Bayesian Optimization) ---


[I 2026-04-23 04:59:55,118] Trial 1 finished with value: 1.0 and parameters: {'max_depth': 11, 'min_samples_split': 4}. Best is trial 0 with value: 1.0.
[I 2026-04-23 04:59:55,190] Trial 2 finished with value: 1.0 and parameters: {'max_depth': 12, 'min_samples_split': 15}. Best is trial 0 with value: 1.0.
[I 2026-04-23 04:59:55,250] Trial 3 finished with value: 1.0 and parameters: {'max_depth': 6, 'min_samples_split': 4}. Best is trial 0 with value: 1.0.
[I 2026-04-23 04:59:55,300] Trial 4 finished with value: 1.0 and parameters: {'max_depth': 8, 'min_samples_split': 15}. Best is trial 0 with value: 1.0.
[I 2026-04-23 04:59:55,331] Trial 5 finished with value: 1.0 and parameters: {'max_depth': 32, 'min_samples_split': 20}. Best is trial 0 with value: 1.0.
[I 2026-04-23 04:59:55,363] Trial 6 finished with value: 1.0 and parameters: {'max_depth': 2, 'min_samples_split': 14}. Best is trial 0 with value: 1.0.
[I 2026-04-23 04:59:55,391] Trial 7 finished with value: 1.0 and parameters: {'ma

Optuna Best Params: {'max_depth': 5, 'min_samples_split': 8}

--- Final Model Evaluation (5-Fold CV) ---
Mean Accuracy: 1.0000
Standard Deviation: 0.0000
